In [2]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio. 
# Si no lo tienes a la mano, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "https://raw.githubusercontent.com/mricardo89/data-mining/refs/heads/main/Unit-III/datasets/players_data-2025_2026.csv"
df = pd.read_csv(url)


FASE 1 : Construyendo el Multiverso del Jugador (Min. 4 Dimensiones)

1. Construyan la matriz multidimensional X seleccionando al menos 4 características numéricas (pueden ser más) continuas (float) que definan el rendimiento del jugador (por ejemplo: Goles, Asistencias, Minutos_Jugados, Precisión_Pases, Recuperaciones, etc.).

In [ ]:
# Build the multidimensional feature matrix X
# The source CSV does not include xAG/PrgP/KP, so we use the available continuous
# performance metrics present in the dataset.

feature_cols = ['Gls', 'Ast', 'G+A', 'Int', 'Crs', 'TklW']
X_matrix = df[feature_cols].dropna().to_numpy(dtype=float)

df_X = pd.DataFrame(X_matrix, columns=feature_cols)
print("Shape:", df_X.shape)
print(df_X.head())

Shape: (2839, 6)
   Gls  Ast  G+A   Int   Crs  TklW
0  4.0  5.0  9.0  17.0  37.0  27.0
1  0.0  0.0  0.0   1.0   0.0   0.0
2  0.0  0.0  0.0   2.0   3.0   4.0
3  2.0  0.0  2.0   3.0  30.0   3.0
4  0.0  0.0  0.0   2.0   1.0   1.0


2. Creen un vector para su jugador estrella y otro vector para el resto de los candidatos.

In [ ]:
# ejemplo: si tu jugador estrella es "Pedri"
jugador_estrella = "Pedri"

# 1) Identifica la fila del jugador estrella en el DataFrame original
idx_star = df.index[df["Player"] == jugador_estrella][0]

# 2) Calcula los z-scores sobre todas las filas válidas
media = X_matrix.mean(axis=0, keepdims=True)
desviacion = X_matrix.std(axis=0, keepdims=True)
X_zscore = (X_matrix - media) / desviacion

# 3) Vector del jugador estrella
vector_pedri = X_zscore[df.index.get_loc(idx_star)]

# 4) Resto de candidatos
mask_restantes = np.ones(len(X_zscore), dtype=bool)
mask_restantes[df.index.get_loc(idx_star)] = False

resto_candidatos = X_zscore[mask_restantes]

3. Pregunta de control: Antes de medir cualquier distancia, ¿qué operación de la Unidad II deben aplicarle obligatoriamente a estas 4 columnas para que los Minutos_Jugados (rango en miles) no aplasten matemáticamente a los Goles (rango en decenas)? Apliquen esa transformación en el código.

Lo que se debe realizar es una tecnica de escalamiento para evitar el desborde matematico al momento de realizar las predicciones, por lo que es necesario aplicar en este caso el Z-Score para obtener una estandarizavion correcta de los valores de las stats.


Fase 2: El Duelo de Distancias (Euclidiana vs. Manhattan)

Programen una celda para encontrar al jugador más similar a su estrella usando la Distancia Euclidiana. Impriman el nombre del reemplazo sugerido.

In [6]:
# Distancia Euclidiana para encontrar al jugador más similar a la estrella

candidate_indices = np.where(mask_restantes)[0]

# Distancias euclidianas entre el vector de la estrella y cada candidato
distancias = np.linalg.norm(resto_candidatos - vector_pedri, axis=1)

# Índice del candidato más cercano
best_local_idx = np.argmin(distancias)
best_original_idx = candidate_indices[best_local_idx]

# Nombre del reemplazo sugerido
jugador_sugerido = df.loc[best_original_idx, "Player"]

print(f"Jugador estrella: {jugador_estrella}")
print(f"Reemplazo sugerido (Distancia Euclidiana): {jugador_sugerido}")

Jugador estrella: Pedri
Reemplazo sugerido (Distancia Euclidiana): Konrad Laimer


En la siguiente celda, busquen al reemplazo calculando la Distancia Manhattan (deberán generar una función custom para calcular esta distancia).

In [7]:
# Distancia Manhattan para encontrar al jugador más similar a la estrella

def manhattan_distance(a, b):
    # a y b pueden ser arrays 1D o 2D; calcula la suma de diferencias absolutas
    return np.abs(a - b).sum(axis=1)


candidate_indices = np.where(mask_restantes)[0]

distancias_manhattan = manhattan_distance(resto_candidatos, vector_pedri)

best_local_idx_manhattan = np.argmin(distancias_manhattan)
best_original_idx_manhattan = candidate_indices[best_local_idx_manhattan]

# Nombre del reemplazo sugerido
jugador_sugerido_manhattan = df.loc[best_original_idx_manhattan, "Player"]

print(f"Jugador estrella: {jugador_estrella}")
print(f"Reemplazo sugerido (Distancia Manhattan): {jugador_sugerido_manhattan}")
print(f"Distancia Manhattan mínima: {distancias_manhattan[best_local_idx_manhattan]}")

Jugador estrella: Pedri
Reemplazo sugerido (Distancia Manhattan): Konrad Laimer
Distancia Manhattan mínima: 2.5632016248903806


Análisis Crítico: ¿El algoritmo sugirió al mismo jugador en ambos casos? Como ingenieros, expliquen en un bloque de Markdown las diferencias observables entre la Distancia Euclidiana comparado con la Distancia Manhattan.
